# 06 — Diagnóstico: sinal espacial imagem-texto no LIBERO

Antes de considerar usar um modelo congelado como um "gate" espacial de
linguagem (uma 4a abordagem de fusão, sem nenhum parâmetro novo treinável,
além de Token/FiLM/CrossAttention já implementadas em
`src/act_lang/models/fusion/`), este notebook verifica com dados reais do
LIBERO se o mapa de relevância imagem-texto de cada método é
(a) espacialmente concentrado no objeto/ação certo e (b) **muda quando a
instrução muda**.

Compara **quatro métodos**, do mais fraco ao mais forte sinal espacial
esperado: **SigLIP 1**, **SigLIP 2**, **CLIPSeg** e **Grounded SAM**
(Grounding DINO + SAM). Os dois primeiros são só similaridade de cosseno
patch-a-patch (treino contrastivo sobre um vetor global, sem supervisão
espacial); os dois últimos têm supervisão de localização de verdade no
treino (caixa/máscara pareada com texto) — deveriam dar um sinal mais limpo,
mas Grounded SAM é bem mais pesado (dois modelos encadeados) e não é mais
"zero-parâmetro barato" no sentido original — ver seção 5.

**Risco conhecido** (motivo de rodar isso antes de integrar no pipeline):
SigLIP/CLIP são treinados com uma loss contrastiva sobre um vetor
**global** (pooled), não patch a patch — literatura de dense-prediction
zero-shot (MaskCLIP, GEM) mostra que a similaridade crua por patch costuma
ser ruidosa sem ajustes extras. Imagens de câmera de robô em simulação
também são fora da distribuição de todos esses modelos (treinados em
dados/fotos web). Sendo zero-parâmetro (SigLIP/CLIPSeg) ou pelo menos sem
fine-tuning no LIBERO (Grounded SAM), nenhum desses métodos tem como se
corrigir via gradiente se o sinal vier ruim — por isso vale validar antes
de investir na integração completa.

**O que este notebook faz, na ordem:**
1. Clona o repo e instala as dependências (`hdf5` + `vlm`)
2. Aponta pro dataset HDF5 (baixa se ainda não tiver nesta sessão — mesmo
   caminho `/content/libero_hdf5` do `04_treino_hdf5.ipynb`, reaproveita se
   já baixado)
3. (Opcional) valida isolado o Grounded SAM, a parte de API mais incerta
4. Roda `scripts/diagnose_siglip_patch_alignment.py`, que salva PNGs
   comparando, pra cada método, o heatmap da instrução certa vs. uma
   instrução errada, e imprime um resumo numérico por método
5. Mostra os PNGs inline

Puramente exploratório — não treina nada, não toca no modelo ACT nem em
`fusion/`. Não precisa de GPU (todos os métodos são frozen inference — leve
o suficiente pra rodar até em CPU, mais lento pro Grounded SAM), mas usa se
disponível.

## 1. Repositório e ambiente

In [ ]:
!git clone -b teste https://github.com/rafaelheydt/act-lang.git
%cd act-lang

# "hdf5" traz h5py + huggingface_hub (leitura do dataset nativo); "vlm" traz
# transformers, novo, só pra este diagnóstico (SiglipModel/SiglipProcessor).
!pip install -q -e ".[hdf5,vlm]"

import sys
sys.path.insert(0, "src")
sys.path.insert(0, ".")


## 2. Dataset HDF5

Baixa só se `DATA_DIR` ainda não existir nesta sessão (reaproveita se você
já rodou o `04_treino_hdf5.ipynb` antes). `PILOT_LIMIT` pequeno é suficiente
aqui — só precisamos de poucas imagens+instruções, não do dataset inteiro.

In [ ]:
PILOT_LIMIT = 3  # None = as 40 tarefas completas (~28-32GB); aqui só precisamos de poucas amostras
DATA_DIR = "/content/libero_hdf5"  # mesmo caminho do 04_treino_hdf5.ipynb

import subprocess, sys
from pathlib import Path

if Path(DATA_DIR).is_dir() and any(Path(DATA_DIR).iterdir()):
    print(f"{DATA_DIR} já existe e não está vazio -- pulando download (reaproveitando desta sessão)")
else:
    cmd = [sys.executable, "-u", "scripts/download_libero_hdf5.py", "--out", DATA_DIR]
    if PILOT_LIMIT is not None:
        cmd += ["--limit", str(PILOT_LIMIT)]
    print("rodando:", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print("
código de saída:", proc.returncode)
    assert proc.returncode == 0, "download falhou -- veja o log acima"


## 3. Diagnóstico: mapa de relevância imagem-texto

Pra cada uma de `N_SAMPLES` amostras (espalhadas pelo dataset, não só os
primeiros frames): roda os métodos configurados, calcula o mapa de
relevância espacial pra instrução **certa** e pra uma instrução **errada**
(de outra tarefa do dataset), salva os dois lado a lado em PNG.

Por padrão compara **quatro métodos**, do mais fraco ao mais forte sinal
espacial esperado: SigLIP 1, SigLIP 2, CLIPSeg e Grounded SAM (Grounding
DINO + SAM). Os dois últimos têm supervisão de localização de verdade no
treino (caixa/máscara pareada com texto) — deveriam dar sinal mais limpo,
mas Grounded SAM é bem mais pesado (dois modelos encadeados, download extra
de ~1.8GB) e não é mais "zero-parâmetro barato" no sentido original.

Requer mais de uma tarefa distinta no `DATA_DIR` pro teste de
discriminabilidade fazer sentido (o pré-requisito `PILOT_LIMIT >= 2` acima
garante isso).

**Primeira vez rodando:** vale validar só a parte mais incerta antes de
rodar tudo — descomente a célula de validação isolada do Grounded SAM logo
abaixo (`--n-samples 1 --skip-siglip --skip-clipseg`) antes da célula
principal.

### (Opcional, primeira vez) validar só o Grounded SAM isolado

A parte de API menos certa do script é o encadeamento Grounding DINO + SAM
(ver nota no cabeçalho do script). Rodar isolado com 1 amostra antes do
resto ajuda a pegar erro de API cedo, sem esperar os outros modelos
carregarem.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-u", "scripts/diagnose_siglip_patch_alignment.py",
    "--hdf5-dir", DATA_DIR,
    "--n-samples", "1",
    "--skip-siglip", "--skip-clipseg",
    "--output-dir", "siglip_patch_alignment_outputs_validacao",
]
print("rodando:", " ".join(cmd))
subprocess.run(cmd, check=False)


In [ ]:
N_SAMPLES = 6
OUTPUT_DIR = "siglip_patch_alignment_outputs"
# Default do script já compara SigLIP 1, SigLIP 2, CLIPSeg e Grounded SAM.
# Pra rodar só um subconjunto (mais rápido, ex. iterando), descomente:
# EXTRA_ARGS = ["--skip-grounded-sam"]  # ou --skip-siglip / --skip-clipseg
EXTRA_ARGS = []

import subprocess, sys
from pathlib import Path
from datetime import datetime

Path("logs").mkdir(exist_ok=True)
log_path = f"logs/siglip_diag_{datetime.now():%Y%m%d_%H%M}.log"

cmd = [
    sys.executable, "-u", "scripts/diagnose_siglip_patch_alignment.py",
    "--hdf5-dir", DATA_DIR,
    "--n-samples", str(N_SAMPLES),
    "--output-dir", OUTPUT_DIR,
] + EXTRA_ARGS
print("rodando:", " ".join(cmd))
print("log em:", log_path, "
" + "=" * 60)

with open(log_path, "a") as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

print("
" + "=" * 60)
print(f"processo encerrado com código {proc.returncode}"
      + (" -- verifique o log acima" if proc.returncode != 0 else " -- ok"))


## 4. Heatmaps

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

for path in sorted(glob.glob(f"{OUTPUT_DIR}/*.png")):
    plt.figure(figsize=(13, 4.5))
    plt.imshow(Image.open(path))
    plt.axis("off")
    plt.title(path)
    plt.show()


### Salvar os PNGs (zip + download)

Compacta `OUTPUT_DIR` e baixa pro seu computador (no Colab) -- o disco
`/content` é apagado quando a sessão termina, então isso é o jeito de levar
os heatmaps pra fora antes de fechar. Rodando localmente, os PNGs já estão
em disco, então só avisa onde.

In [ ]:
import shutil

from act_lang.utils.runtime import is_colab

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("zip criado em:", zip_path)

if is_colab():
    from google.colab import files
    files.download(zip_path)
else:
    print(f"rodando localmente -- os PNGs já estão em {OUTPUT_DIR}/, sem necessidade de download")


## 5. Como interpretar

Olhe tanto os heatmaps acima quanto o resumo numérico impresso pela célula
principal da seção 3 (`diff médio`, um valor por método):

- **Heatmap da instrução certa concentrado no objeto/região certa, e
  visivelmente diferente do heatmap da instrução errada** → o sinal existe e
  é específico da linguagem pra aquele método. Se for SigLIP/CLIPSeg, vale a
  pena desenhar a integração completa no pipeline (um módulo que substitui
  `VisionBackbone` + `fuse()`, já que o mapa só faz sentido calculado no
  espaço/resolução nativa do método). Se for só o Grounded SAM que vencer
  claramente, a decisão muda de "gate barato" pra "vale o custo de
  engenharia de um pipeline de detecção completo dentro do ACT?" — ele já
  não é zero-parâmetro no sentido original.
- **Heatmaps parecidos entre instrução certa/errada, ou `diff médio` perto
  de zero, em todos os métodos** → confirma o risco: nem o sinal cru
  (SigLIP) nem os métodos com supervisão de localização (CLIPSeg, Grounded
  SAM) discriminam bem nas imagens do LIBERO — a ideia de gate zero-
  parâmetro provavelmente não vale a pena sem alguma forma de calibração/
  fine-tuning (ver ReSiReg/CLIP-DINOiser como caminhos de correção
  publicados). Reconsiderar as outras opções discutidas: SigLIP como
  backbone + fusão aprendida (Token/FiLM/CrossAttention), ou um VLM
  totalmente fundido tipo PaliGemma.